<a href="https://colab.research.google.com/github/iamajeet/colab-workbook/blob/main/agentic_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
# ============================================
# API KEY: Get your key from https://aicafe.hcl.com/AICafe/#/tutorials/api-docs
# IMPORTANT: Enter your API key in the GUI instead of hardcoding it here
# ============================================
API_KEY = "d1abf648-8075-4df2-b431-0549d6d8866e"  # <-- Use the GUI to enter your key
# --- Endpoint config ---
API_URL = (
    "https://aicafe.hcl.com/AICafeService/api/v1/subscription/openai/"
    "deployments/gpt-4.1/chat/completions?api-version=2024-12-01-preview"
)
SYSTEM_PROMPT = """
You are a helpful AI agent.
- Answer clearly and concisely.
- Ask for clarification only when truly needed.
"""
def run_agent():
    print("Simple AI agent. Type 'quit' to exit.\n")
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in {"quit", "exit"}:
            print("Agent: Goodbye! 👋")
            break
        # Build request body (OpenAI-style)
        body = {
            "model": "gpt-4.1",
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_input},
            ],
            "temperature": 0.7,
        }
        headers = {
            "api-key": API_KEY,
            "Content-Type": "application/json",
            "Accept": "application/json",
        }
        try:
            response = requests.post(API_URL, headers=headers, json=body, timeout=60)
            response.raise_for_status()
            data = response.json()
            agent_reply = data["choices"][0]["message"]["content"].strip()
            print(f"Agent: {agent_reply}\n")
        except Exception as e:
            print(f"Error: {e}")
if __name__ == "__main__":
    run_agent()


In [ ]:
!pip install langchain langchain-community langchain-google-genai wikipedia


In [1]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

In [2]:
from google.colab import userdata
from google import genai
# Load API key from Colab Secrets into environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [3]:
# Set your Gemini API key
#os.environ["GOOGLE_API_KEY"] = "your_gemini_api_key"

# Create Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# Create Wikipedia tool
wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

# Ask user question
query = input("Ask a question: ")

# Retrieve Wikipedia content
wiki_result = wiki.run(query)
print("\nWikipedia result:")
print(wiki_result)

# Send to Gemini for summarization
prompt = f"""
Answer the question using the information from Wikipedia.

Question: {query}

Wikipedia information:
{wiki_result}

Provide a clear and concise answer.
"""

Ask a question: who is sania mirza

Wikipedia result:
Page: Sania Mirza
Summary: Sania Mirza ([ˈsaːnijaː ˈmirzaː]; born 15 November 1986) is an Indian former professional tennis player. A former doubles world No. 1, she won six major titles – three in women's doubles and three in mixed doubles. From 2003 until her retirement from singles in 2013, she was ranked by the Women's Tennis Association as the No. 1 Indian in singles. Throughout her career, Mirza has established herself as one of the most known, highest-paid, and influential athletes in India.

In singles, Mirza had wins over Svetlana Kuznetsova, Vera Zvonareva, and Marion Bartoli, as well as former world-number-ones Martina Hingis, Dinara Safina, and Victoria Azarenka. She is the highest-ranked Indian female player ever, peaking at world No. 27 in mid-2007. However, a major wrist injury caused her to shift to doubles. Mirza has achieved a number of firsts for women's tennis in India, including reaching the one million-US$ mark

In [4]:

response = llm.invoke(prompt)

print("\nAnswer:")
print(response.content)


Answer:
Sania Mirza is an Indian former professional tennis player, born on 15 November 1986. She was a former doubles world No. 1 and won six major titles (three in women's doubles and three in mixed doubles). Throughout her career, she established herself as one of the most known, highest-paid, and influential athletes in India. Mirza retired from professional tennis in February 2023.


In [5]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun
from langchain_core.messages import HumanMessage


import os
from google.colab import userdata
from google import genai
# Load API key from Colab Secrets into environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [6]:
# Set Gemini API Key
# os.environ["GOOGLE_API_KEY"] = "your_gemini_api_key"

# Create Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# Wikipedia tool
wiki_api = WikipediaAPIWrapper()
wiki_tool = WikipediaQueryRun(api_wrapper=wiki_api)

# Function to answer questions
def ask_agent(question):

    # Step 1: search wikipedia
    wiki_result = wiki_tool.run(question)

    # Step 2: send to Gemini
    prompt = f"""
    Answer the user's question using the Wikipedia information below.

    Question:
    {question}

    Wikipedia information:
    {wiki_result}

    Give a clear and short answer.
    """

    response = llm.invoke([HumanMessage(content=prompt)])

    return response.content



In [7]:
# Interactive loop
while True:
    query = input("\nAsk a question (type exit to stop): ")

    if query.lower() == "exit":
        break

    answer = ask_agent(query)

    print("\nAnswer:", answer)


Ask a question (type exit to stop): who is john abraham

Answer: John Abraham is an Indian actor, writer, and film producer who primarily works in Hindi films. He was also a former model.

Ask a question (type exit to stop): exit
